In [1]:
import torch 


# input of layer 2
bs = 16 # batch size

Leye = torch.rand(bs,128, 8, 8)
print(Leye)
Reye = torch.rand(bs,128, 8, 8)
print(Reye)
FaceData = torch.rand(bs,128, 8, 8)  # currently matched with eyes......
print(FaceData)

tensor([[[[0.0361, 0.5637, 0.6002,  ..., 0.4950, 0.8424, 0.7683],
          [0.3119, 0.1607, 0.2325,  ..., 0.0588, 0.3152, 0.8403],
          [0.9468, 0.8308, 0.1701,  ..., 0.4843, 0.6776, 0.2019],
          ...,
          [0.6287, 0.0380, 0.0079,  ..., 0.1675, 0.4175, 0.9097],
          [0.3107, 0.6158, 0.3821,  ..., 0.3963, 0.3356, 0.3909],
          [0.0484, 0.6075, 0.9110,  ..., 0.9622, 0.8089, 0.7468]],

         [[0.8107, 0.1685, 0.3582,  ..., 0.1789, 0.7123, 0.1245],
          [0.1604, 0.8244, 0.6180,  ..., 0.2623, 0.8416, 0.1104],
          [0.3589, 0.2699, 0.0406,  ..., 0.5989, 0.2971, 0.1924],
          ...,
          [0.9265, 0.2900, 0.0586,  ..., 0.1664, 0.7964, 0.3349],
          [0.0193, 0.8593, 0.7265,  ..., 0.9142, 0.4172, 0.5707],
          [0.1069, 0.5707, 0.0356,  ..., 0.9042, 0.2992, 0.2710]],

         [[0.5651, 0.3812, 0.4839,  ..., 0.4235, 0.5308, 0.7397],
          [0.1676, 0.3901, 0.3967,  ..., 0.2020, 0.4405, 0.6323],
          [0.5449, 0.8577, 0.9716,  ..., 0

In [13]:
from vit_pytorch import ViT


class TinyModel(torch.nn.Module):
    def __init__(self, gaze_dims=2):    # the gaze_dims should be defined later, according to the dataset and what we wanna predict
        super(TinyModel, self).__init__()
        # layer 1: feature extraction (to be implemented)
        
        
        # layer 2: feature fusion: concate + group  normalization
        # self.concate = torch.cat((x, x, x), 0) // not needed here, only in forward
        # total_ch = Leye[1]+Reye[1]+FaceData[1]  # total channels of the input // this is not working 
        total_ch = 384
        self.gn = torch.nn.GroupNorm(3, total_ch)     # Separate 6 channels into 3 groups, how to define a appropriate number of groups & channels?
        
        # layer 3:self-attention
        self.vit = ViT(
                image_size = 8,
                patch_size = 2,
                num_classes = 3,  # for instance 3 gaze dims, PoG (x,y,z)
                dim = 1024,
                depth = 6,
                heads = 16,
                mlp_dim = 2048,
                channels= total_ch,
                dropout = 0.1,
                emb_dropout = 0.1)
        
        # can replace the mlp_head with a new one for regression tasks, or just simply change the num_classes to the number of gaze dims
        # self.vit.mlp_head = torch.nn.Sequential(
        #     torch.nn.Linear(1024, 512),
        #     torch.nn.ReLU(),
        #     torch.nn.Dropout(0.1),
        #     torch.nn.Linear(512, 256),
        #     torch.nn.ReLU(),
        #     torch.nn.Dropout(0.1),
        #     torch.nn.Linear(256, gaze_dims)
        # )


    def forward(self, left_eye, right_eye, face):

        # layer2
        concate = torch.cat((left_eye, right_eye, face), 1)  # dim = 0 or 1?  only channel dim changes?
        out = self.gn(concate)
        bs, c, h, w = out.shape
        out = out.reshape(bs, c, h, w)
        out = self.vit(out)
        return out 
        

model = TinyModel(gaze_dims=3)
print(model)

output = model(Leye, Reye, FaceData)  # currently the face Data is not matched with eyes, should be matched in the future, how to match? linear padding or other methods? would it affect the performance if change the size of facedata
print("output:",output.shape)
# testing only forward pass, not backward pass


TinyModel(
  (gn): GroupNorm(3, 384, eps=1e-05, affine=True)
  (vit): ViT(
    (to_patch_embedding): Sequential(
      (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=2, p2=2)
      (1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
      (2): Linear(in_features=1536, out_features=1024, bias=True)
      (3): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    )
    (dropout): Dropout(p=0.1, inplace=False)
    (transformer): Transformer(
      (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): Attention(
            (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attend): Softmax(dim=-1)
            (dropout): Dropout(p=0.1, inplace=False)
            (to_qkv): Linear(in_features=1024, out_features=3072, bias=False)
            (to_out): Sequential(
              (0): Linear(in_features=1024, out_features=1024, bias=True)
              (1): Dropou